In [ ]:
import torch
print("device_count:", torch.cuda.device_count())
print("is_available:", torch.cuda.is_available())

In [ ]:
import sys, site
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig
from datasets import Dataset
import json

In [ ]:
with open("train.json", "r") as f:
    train_dataset = Dataset.from_list(json.load(f))
with open("validation.json", "r") as f:
    validation_dataset = Dataset.from_list(json.load(f))
print(f"Train: {len(train_dataset)} examples, Validation: {len(validation_dataset)} examples")

In [ ]:
def basic_formatting_function(row):
    import json
    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    with open(f"./schemas/{clean_id}.json") as f:
        schema_data = json.load(f)
    schema = {}
    for table_name in schema_data["table_names_original"]:
        schema[table_name] = []
    for i, name in schema_data["column_names_original"]:
        if i == -1:
            continue
        schema[schema_data["table_names_original"][i]].append(name)
    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists."
    )
    prompt = (
        f"Database schema: {schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return a JSON object with only the relevant tables as keys and lists of relevant column names as values. "
        "You MUST include specific column names — do not return empty lists unless a table has no relevant columns. "
        "Example: {\"Orders\": [\"order_id\", \"total\"], \"Customers\": [\"name\"]}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {"text": f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{answer}<|im_end|>"}


In [ ]:
experiment = Experiment(experiment_name="augmented_smollm2_top3_experiment_rf", mode="fit")

In [ ]:
import torch

QWEN = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
ATTN = ["q_proj", "k_proj", "v_proj", "o_proj"]
ALL = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

configs_spec = [
    ("Q5_r4_attn_lr1e5_cosine_e3", basic_formatting_function, 4, 8, 1e-5, ATTN, "cosine", 3),
    ("P1_r4_attn_lr1e5_e2", basic_formatting_function, 4, 8, 1e-5, ATTN, "linear", 2),
    ("P2_r4_attn_lr1e5_e3", basic_formatting_function, 4, 8, 1e-5, ATTN, "linear", 3),
]

all_configs = []
for label, fmt_func, r, alpha, lr, target_modules, scheduler, epochs in configs_spec:
    all_configs.append(RFModelConfig(
        model_name=QWEN,
        peft_config=RFLoraConfig(r=r, lora_alpha=alpha, lora_dropout=0.1, target_modules=target_modules, bias="none"),
        training_args=RFSFTConfig(
            learning_rate=lr,
            lr_scheduler_type=scheduler,
            num_train_epochs=epochs,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=4,
            gradient_checkpointing=False,
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=20,
            bf16=False,
            fp16=True,
        ),
        model_type="causal_lm",
        model_kwargs={"torch_dtype": torch.float16, "use_cache": False},
        formatting_func=fmt_func,
    ))

print(f"Total configs: {len(all_configs)}")
for i, (label, *_) in enumerate(configs_spec):
    print(f"  [{i+1}] {label}")
config_set = List(all_configs)

In [ ]:
def sample_create_model(model_config):
    import gc, os
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model_kwargs = dict(model_config["model_kwargs"])
    model_kwargs.pop("device_map", None)
    model_kwargs.setdefault("low_cpu_mem_usage", True)
    model = AutoModelForCausalLM.from_pretrained(model_config["model_name"], **model_kwargs)
    tokenizer = AutoTokenizer.from_pretrained(model_config["model_name"])
    return (model, tokenizer)

In [ ]:
config_group = RFGridSearch(configs=config_set, trainer_type="SFT")

In [ ]:
experiment.run_fit(
    config_group,
    sample_create_model,
    train_dataset,
    validation_dataset,
    num_chunks=1,
    seed=42,
)

In [ ]:
experiment.end()